# Gans Scooter Demand Pipeline: Weather, Airports and Flight Arrivals

## Project Overview

This notebook presents a complete data pipeline developed for Gans scooter demand management.

The objective is to combine multiple data sources to support scooter positioning decisions:

- City location data (latitude and longitude)
- Current weather conditions from the OpenWeather API
- Airport information and arriving flight data from the AeroDataBox API (via RapidAPI)

Weather conditions influence scooter usage, while airport arrivals provide an indicator of potential customer demand.

By combining both data sources, Gans can make better decisions about where and when to position scooters.

The pipeline:
1. Extracts data from external APIs
2. Transforms API responses into structured Pandas DataFrames
3. Loads processed datasets into a MySQL relational database

## Data Sources

### OpenWeather API
Provides current weather information:
- Temperature
- Feels-like temperature
- Humidity
- Weather conditions
- Wind speed

### AeroDataBox API (via RapidAPI)
Provides airport and flight arrival information:
- Airport details
- IATA and ICAO codes
- Flight numbers
- Airlines
- Origin and destination airports
- Arrival times
- Flight status

## Business Objective

The final goal is to combine:

City Location
+
Weather Conditions
+
Airport Arrivals

to support scooter demand estimation and positioning decisions.


---
## 1. Import libraries 💾

In [ ]:

import pandas as pd
import requests
import json
import getpass

from bs4 import BeautifulSoup
from getpass import getpass
from sqlalchemy import create_engine, text


---
## 2. Database connection 🔌


Change the values below to match your MySQL setup.

If you are working locally in JupyterLab, `host = "127.0.0.1"` is usually correct.

If you are using Google Colab, your MySQL database must be available online. A local MySQL database on your laptop cannot normally be reached using `127.0.0.1` from Colab.


In [ ]:

schema = "gans"
host = "127.0.0.1"
user = "root"
port = 3306

password = getpass.getpass("Enter your MySQL password: ")

connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"


Test the connection before continuing.

In [ ]:

engine = create_engine(connection_string)

with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Connection successful:", result.fetchone())


---
## 3. Create the database tables 🗂️


This creates three tables if they do not already exist:

- `cities`
- `weather_current`
- `scooter_weather_forecast`

If you already created these tables in MySQL Workbench, this cell should still be safe because it uses `CREATE TABLE IF NOT EXISTS`.


---
## 4. Choose the cities to scrape 🌍


The dictionary below contains the cities we want to collect.

You can add or remove cities by editing this dictionary.


In [ ]:

city_pages = {
    "Berlin": {
        "country": "Germany",
        "wiki_url": "https://en.wikipedia.org/wiki/Berlin"
    },
    "Hamburg": {
        "country": "Germany",
        "wiki_url": "https://en.wikipedia.org/wiki/Hamburg"
    },
    "Munich": {
        "country": "Germany",
        "wiki_url": "https://en.wikipedia.org/wiki/Munich"
    },
    "Cologne": {
        "country": "Germany",
        "wiki_url": "https://en.wikipedia.org/wiki/Cologne"
    },
    "Frankfurt": {
        "country": "Germany",
        "wiki_url": "https://en.wikipedia.org/wiki/Frankfurt"
    }
}

city_pages


---
## 5. Scrape latitude and longitude from Wikipedia 🕸️


Wikipedia pages often contain a hidden `geo` tag with coordinates in decimal format.

Example:

```text
52.52000; 13.40500
```

We can scrape this value, split it into latitude and longitude, and store it in a DataFrame.


In [ ]:

city_data = []

for city_name, city_info in city_pages.items():

    url = city_info["wiki_url"]
    country = city_info["country"]

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    soup = BeautifulSoup(response.content, "html.parser")

    geo_tag = soup.find("span", class_="geo")

    if geo_tag is not None:
        coordinates = geo_tag.get_text()
        latitude = coordinates.split(";")[0].strip()
        longitude = coordinates.split(";")[1].strip()

        city_data.append({
            "city_name": city_name,
            "country": country,
            "latitude": latitude,
            "longitude": longitude,
            "wiki_url": url,
            "date_retrieved": pd.Timestamp.now()
        })

    else:
        print("Coordinates not found for:", city_name)

city_data


Create a DataFrame from the scraped city data.

In [ ]:

cities_df = pd.DataFrame(city_data)

cities_df


Check the data types.

In [ ]:

cities_df.info()


Convert latitude and longitude to numeric values.

In [ ]:

cities_df["latitude"] = pd.to_numeric(cities_df["latitude"])
cities_df["longitude"] = pd.to_numeric(cities_df["longitude"])

cities_df.info()


---
## 6. Send the city data to SQL 📠


The `cities` table has an automatic `city_id`, so we do not include a `city_id` column in the DataFrame.

Be careful with `if_exists="append"`: if you run this cell many times, you may insert the same cities more than once.


In [ ]:
pd.read_sql("""
    DESCRIBE cities
""", con=connection_string)

In [ ]:
cities_to_sql = cities_df[
    [
        "city_name",
        "country",
        "latitude",
        "longitude",
        "country_code",
        "city_timezone",
        "date_retrieved"
    ]
]

In [ ]:
cities_to_sql.to_sql(
    "cities",
    con=connection_string,
    if_exists="append",
    index=False
)

In [ ]:



cities_df.to_sql(
    "cities",
    con=connection_string,
    if_exists="append",
    index=False
)


In [ ]:
cities_df["country_code"] = "DE"
cities_df["city_timezone"] = "Europe/Berlin"

Read the cities back from SQL to check that they were inserted correctly.

In [ ]:

cities_from_sql = pd.read_sql("""
    SELECT city_id,
           city_name,
           country,
           latitude,
           longitude
    FROM cities
""", con=connection_string)

cities_from_sql


---
## 7. Request current weather from OpenWeather 🌦️


We now use the latitude and longitude from the SQL table to make requests to OpenWeather.

The endpoint used here is the one you provided:

```text
https://api.openweathermap.org/data/4.0/onecall/current?lat={lat}&lon={lon}&appid={API key}
```

This notebook also adds:

```text
units=metric
```

so the temperature is returned in Celsius.


In [ ]:

openweather_api_key = getpass.getpass("Enter your OpenWeather API key: ")


In [ ]:
# Use the first city from your SQL table as a test

test_city = cities_from_sql.iloc[0]

lat = test_city["latitude"]
lon = test_city["longitude"]

url = (
    "https://api.openweathermap.org/data/2.5/weather"
    f"?lat={lat}"
    f"&lon={lon}"
    "&units=metric"
    f"&appid={openweather_api_key}"
)

weather_request = requests.get(url)

print("Status code:", weather_request.status_code)

weather_json = weather_request.json()

weather_json

In [ ]:

current_weather = weather_json["data"][0]

current_weather.keys()


In [ ]:
temperature = weather_json["main"]["temp"]
feels_like = weather_json["main"]["feels_like"]
humidity = weather_json["main"]["humidity"]
weather_main = weather_json["weather"][0]["main"]
weather_description = weather_json["weather"][0]["description"]
wind_speed = weather_json["wind"]["speed"]

print("Temperature:", temperature)
print("Feels like:", feels_like)
print("Humidity:", humidity)
print("Weather:", weather_main)
print("Description:", weather_description)
print("Wind speed:", wind_speed)

---
## 8. Loop over all cities and build a weather DataFrame 🔁

In [ ]:
weather_data = []

for index, city in cities_from_sql.iterrows():

    lat = city["latitude"]
    lon = city["longitude"]

    url = (
        "https://api.openweathermap.org/data/2.5/weather"
        f"?lat={lat}"
        f"&lon={lon}"
        "&units=metric"
        f"&appid={openweather_api_key}"
    )

    response = requests.get(url)

    print(
        city["city_name"],
        response.status_code
    )

    data = response.json()

    if response.status_code == 200:

        weather_data.append({
            "city_id": city["city_id"],
            "temperature": data["main"]["temp"],
            "feels_like": data["main"]["feels_like"],
            "humidity": data["main"]["humidity"],
            "weather_main": data["weather"][0]["main"],
            "weather_description": data["weather"][0]["description"],
            "wind_speed": data["wind"]["speed"],
            "date_retrieved": pd.Timestamp.now()
        })

---
## 9. Send the weather data to SQL 📠

In [ ]:

weather_df = pd.DataFrame(weather_data)

weather_df

Read the weather data back from SQL.

In [ ]:
weather_df.to_sql(
    "weather_current",
    con=connection_string,
    if_exists="append",
    index=False
)


---
## 10. Join cities and weather in SQL 🔗

In [ ]:

city_weather = pd.read_sql("""
    SELECT c.city_name,
           c.country,
           c.latitude,
           c.longitude,
           w.temperature,
           w.feels_like,
           w.humidity,
           w.weather_main,
           w.weather_description,
           w.wind_speed,
           w.date_retrieved
    FROM cities c
    JOIN weather_current w
        ON c.city_id = w.city_id
    ORDER BY w.date_retrieved DESC
""", con=connection_string)

city_weather.head()


---
## 11. Create a simple scooter-management forecast 🛴


This is a simple example of how Gans could use the weather data.

The logic below is intentionally basic:

- Rain, snow, storms or very strong wind → lower scooter demand
- Pleasant temperature and low wind → higher scooter demand
- Everything else → medium demand

This is not a machine-learning forecast. It is a simple rule-based business forecast.


In [ ]:

forecast_data = []

for index, row in city_weather.iterrows():

    weather_text = str(row["weather_description"]).lower()
    temperature = row["temperature"]
    wind_speed = row["wind_speed"]

    if ("rain" in weather_text) or ("snow" in weather_text) or ("storm" in weather_text) or (wind_speed >= 10):
        weather_risk = "High"
        scooter_demand_forecast = "Low"
        notes = "Bad weather may reduce scooter usage."

    elif (temperature >= 15) and (temperature <= 28) and (wind_speed < 7):
        weather_risk = "Low"
        scooter_demand_forecast = "High"
        notes = "Good weather may increase scooter usage."

    else:
        weather_risk = "Medium"
        scooter_demand_forecast = "Medium"
        notes = "Weather conditions are acceptable."

    forecast_data.append({
        "city_id": cities_from_sql.loc[
            cities_from_sql["city_name"] == row["city_name"],
            "city_id"
        ].iloc[0],
        "weather_risk": weather_risk,
        "YOUR_API_KEY": scooter_demand_forecast,
        "notes": notes,
        "date_retrieved": pd.Timestamp.now()
    })

scooter_forecast_df = pd.DataFrame(forecast_data)

scooter_forecast_df.head()


Send the simple scooter forecast to SQL.

Read the final city-weather-forecast view from SQL.

---
## 12. Summary ✅

In [ ]:

final_gans_view = pd.read_sql("""
    SELECT c.city_name,
           c.country,
           w.temperature,
           w.weather_description,
           w.wind_speed,
           f.weather_risk,
           f.scooter_demand_forecast,
           f.notes,
           f.date_retrieved
    FROM cities c
    JOIN weather_current w
        ON c.city_id = w.city_id
    JOIN scooter_weather_forecast f
        ON c.city_id = f.city_id
    ORDER BY f.date_retrieved DESC
""", con=connection_string)

final_gans_view.head()



In this notebook, we completed the full workflow:

1. Scraped city latitude and longitude from Wikipedia.
2. Created a `cities_df` DataFrame.
3. Sent city data to the `cities` SQL table.
4. Read city coordinates back from SQL.
5. Used the coordinates to call the OpenWeather API and AeroDataBox API.
6. Created a `weather_df` DataFrame.
7. Sent weather data to the `weather_current` SQL table.
8. Joined cities and weather in SQL.
9. Created a simple scooter-demand forecast.
10. Stored the result in a SQL table.

This is the core logic for a simple Gans scooter-management weather pipeline.


In [ ]:
airports_from_sql = pd.read_sql("""
    SELECT *
    FROM airports
""", con=connection_string)

airports_from_sql

In [ ]:
cities_from_sql = pd.read_sql("""
    SELECT city_id,
           city_name,
           latitude,
           longitude
    FROM cities
""", con=connection_string)

cities_from_sql

In [ ]:
def get_airports(cities_df):

    headers = {
        "x-rapidapi-key": "YOUR_API_KEY",
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
    }

    all_airports = []

    for index, city in cities_df.iterrows():

        url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

        querystring = {
            "lat": city["latitude"],
            "lon": city["longitude"],
            "radiusKm": "50",
            "limit": "10",
            "withFlightInfoOnly": "true"
        }

        response = requests.get(
            url,
            headers=headers,
            params=querystring
        )

        if response.status_code == 200:

            airports = pd.json_normalize(
                response.json()["items"]
            )

            airports["city_id"] = city["city_id"]

            all_airports.append(airports)

    return pd.concat(all_airports, ignore_index=True)

In [ ]:
airports_raw = get_airports(cities_from_sql)

airports_raw.head()

In [ ]:
airports_raw.columns

In [ ]:
airports_df = airports_raw[
    [
        "city_id",
        "name",
        "iata",
        "icao",
        "location.lat",
        "location.lon"
    ]
].copy()

In [ ]:
airports_df.columns = [
    "city_id",
    "airport_name",
    "iata_code",
    "icao_code",
    "latitude",
    "longitude"
]

airports_df.head()

In [ ]:
airports_df.to_sql(
    "airports",
    con=connection_string,
    if_exists="append",
    index=False
)

In [ ]:
airports_from_sql = pd.read_sql("""
    SELECT airport_id,
           city_id,
           airport_name,
           iata_code
    FROM airports
""", con=connection_string)

airports_from_sql

In [ ]:
airports_df = airports_raw[
    [
        "city_id",
        "name",
        "iata",
        "icao",
        "municipalityName",
        "countryCode",
        "timeZone",
        "location.lat",
        "location.lon"
    ]
].copy()

In [ ]:
airports_df = airports_df.rename(columns={
    "name": "airport_name",
    "iata": "iata_code",
    "icao": "icao_code",
    "municipalityName": "municipality_name",
    "countryCode": "country_code",
    "timeZone": "timezone",
    "location.lat": "latitude",
    "location.lon": "longitude"
})

In [ ]:
airports_df.columns

In [ ]:
airports_df.to_sql(
    "airports",
    con=connection_string,
    if_exists="append",
    index=False
)

In [ ]:
airports_to_sql = airports_df[
    [
        "city_id",
        "airport_name",
        "iata_code",
        "icao_code",
        "latitude",
        "longitude",
        "municipality_name",
        "country_code",
        "timezone"
    ]
]

In [ ]:
airports_to_sql.to_sql(
    "airports",
    con=connection_string,
    if_exists="append",
    index=False
)

In [ ]:
from datetime import datetime, timedelta

tomorrow = datetime.now() + timedelta(days=1)

date = tomorrow.strftime("%Y-%m-%d")

date

---
## 13. AeroDataBox Flight Arrival Pipeline 🛬

Flight arrivals represent potential customer demand around airports.

The pipeline:

1. Reads airports from the SQL database.
2. Generates tomorrow's date automatically.
3. Sends a request to AeroDataBox for arriving flights.
4. Extracts:
   - Flight number
   - Airline
   - Origin airport
   - Destination airport
   - Arrival time
   - Flight status
5. Stores the processed flight data in the SQL `flights` table.

Combining flight arrivals with weather conditions allows Gans to estimate scooter demand.

Example:

- High number of arrivals + good weather → increase scooter availability.
- High number of arrivals + poor weather → adjust allocation.


In [ ]:
import requests

def get_arriving_flights(airports_df):

    headers = {
        "X-RapidAPI-Key": "YOUR_API_KEY",
        "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
    }

    flights = []

    for index, airport in airports_df.iterrows():

        url = (
            f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/"
            f"{airport['iata_code']}/{date}T00:00/{date}T23:59"
        )

        params = {
            "direction": "Arrival"
        }

        response = requests.get(
            url,
            headers=headers,
            params=params
        )

        print(
            airport["iata_code"],
            response.status_code
        )

        if response.status_code == 200:

            data = response.json()

            for flight in data["arrivals"]:

                flights.append({

                    "airport_id": airport["airport_id"],

                    "flight_number": flight.get("number"),

                    "airline": flight.get("airline", {})
                    .get("name"),

                    "origin_airport": flight.get("departure", {})
                    .get("airport", {})
                    .get("name"),

                    "origin_iata": flight.get("departure", {})
                    .get("airport", {})
                    .get("iata"),

                    "scheduled_arrival": flight.get("arrival", {})
                    .get("scheduledTime", {})
                    .get("local"),

                    "estimated_arrival": flight.get("arrival", {})
                    .get("revisedTime", {})
                    .get("local"),

                    "terminal": flight.get("arrival")
                    .get("terminal"),

                    "gate": flight.get("arrival")
                    .get("gate"),

                    "flight_status": flight.get("status"),

                    "date_retrieved": pd.Timestamp.now()
                })

    return pd.DataFrame(flights)

In [ ]:
flights_df = get_arriving_flights(airports_from_sql)

flights_df.head()

In [ ]:
flights_df.to_sql(
    "flights",
    con=connection_string,
    if_exists="append",
    index=False
)

In [ ]:
pd.read_sql("""
    SELECT *
    FROM flights
    LIMIT 10
""", con=connection_string)

In [ ]:
from datetime import datetime, timedelta

tomorrow = datetime.now() + timedelta(days=1)

date = tomorrow.strftime("%Y-%m-%d")

from_time = f"{date}T00:00"
to_time = f"{date}T12:00"

print(from_time)
print(to_time)

In [ ]:
arrival_periods = [
    (f"{date}T00:00", f"{date}T12:00"),
    (f"{date}T12:00", f"{date}T23:59")
]

In [ ]:
for start_time, end_time in arrival_periods:

    url = (
        f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/"
        f"{airport['iata_code']}/{start_time}/{end_time}"
    )

    response = requests.get(
        url,
        headers=headers,
        params={"direction": "Arrival"}
    )

    print(response.status_code)

In [ ]:
url = (
    f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/"
    f"{airport_code}/{from_time}/{to_time}"
)

response = requests.get(
    url,
    headers=headers,
    params={"direction": "Arrival"}
)

print(response.status_code)

In [ ]:
flights_data = []

for flight in response.json()["arrivals"]:

    flights_data.append({

        "airport_id": airport["airport_id"],

        "flight_number": flight.get("number"),

        "airline": flight.get("airline", {}).get("name"),

        "origin_airport": (
            flight.get("departure", {})
            .get("airport", {})
            .get("name")
        ),

        "origin_iata": (
            flight.get("departure", {})
            .get("airport", {})
            .get("iata")
        ),

        "scheduled_arrival": (
            flight.get("arrival", {})
            .get("scheduledTime", {})
            .get("local")
        ),

        "estimated_arrival": (
            flight.get("arrival", {})
            .get("revisedTime", {})
            .get("local")
        ),

        "terminal": (
            flight.get("arrival", {})
            .get("terminal")
        ),

        "gate": (
            flight.get("arrival", {})
            .get("gate")
        ),

        "flight_status": flight.get("status"),

        "date_retrieved": pd.Timestamp.now()
    })


flights_df = pd.DataFrame(flights_data)

flights_df.head()

In [ ]:
flights_df.info()

In [ ]:
flights_data = []

for flight in response.json()["arrivals"]:

    movement = flight.get("movement", {})

    flights_data.append({

        "airport_id": airport["airport_id"],

        "flight_number": flight.get("number"),

        "airline": flight.get("airline", {}).get("name"),

        "origin_airport": (
            movement.get("airport", {})
            .get("name")
        ),

        "origin_iata": (
            movement.get("airport", {})
            .get("iata")
        ),

        "scheduled_arrival": (
            movement.get("scheduledTime", {})
            .get("local")
        ),

        "estimated_arrival": (
            movement.get("revisedTime", {})
            .get("local")
        ),

        "terminal": movement.get("terminal"),

        "gate": movement.get("gate"),

        "flight_status": flight.get("status"),

        "date_retrieved": pd.Timestamp.now()
    })


flights_df = pd.DataFrame(flights_data)

flights_df.head()

In [ ]:
flights_df.info()

In [ ]:
pd.read_sql("""
    DESCRIBE flights
""", con=connection_string)

In [ ]:
flights_df["scheduled_arrival"] = pd.to_datetime(
    flights_df["scheduled_arrival"]
).dt.tz_localize(None)


flights_df["estimated_arrival"] = pd.to_datetime(
    flights_df["estimated_arrival"]
).dt.tz_localize(None)

In [ ]:
flights_df.to_sql(
    "flights",
    con=connection_string,
    if_exists="append",
    index=False
)

In [ ]:
pd.read_sql("""
    SELECT *
    FROM flights
    LIMIT 10
""", con=connection_string)

In [ ]:
flights_df["scheduled_arrival"] = pd.to_datetime(
    flights_df["scheduled_arrival"]
).dt.tz_localize(None)

flights_df["estimated_arrival"] = pd.to_datetime(
    flights_df["estimated_arrival"]
).dt.tz_localize(None)

In [ ]:
flights_df.to_sql(
    "flights",
    con=connection_string,
    if_exists="append",
    index=False
)

In [ ]:
flights_df["destination_airport"] = airport["airport_name"]

flights_df["destination_iata"] = airport["iata_code"]

In [ ]:
flights_df[
    [
        "origin_airport",
        "origin_iata",
        "destination_airport",
        "destination_iata"
    ]
].head()

In [ ]:
flights_df.to_sql(
    "flights",
    con=connection_string,
    if_exists="append",
    index=False
)

In [ ]:
pd.read_sql("""
    SELECT MAX(date_retrieved) AS latest_update
    FROM weather_current
""", con=connection_string)